# Task 2: Calculate INT8 Quantization Parameters

## Objective

Calculate the **scale** and **zero point** required for affine INT8 quantization.

Unlike the previous task where these values were given, this task computes them automatically from the tensor.

## Quantization Range

- q_min = -128
- q_max = 127

## Formula

### Scale

\[
scale=\frac{x_{max}-x_{min}}{q_{max}-q_{min}}
\]

### Zero Point

\[
zero\_point=round\left(q_{min}-\frac{x_{min}}{scale}\right)
\]

Finally,

\[
zero\_point=clip(zero\_point,q_{min},q_{max})
\]

## Edge Cases

- Constant tensor
- Very small floating-point range
- All positive values
- All negative values

In [1]:
import numpy as np

## Step 1: Define Input Tensors

Five tensors are used to test different quantization scenarios.

1. Mixed positive and negative values
2. All positive values
3. All negative values
4. Constant tensor
5. Very small floating-point values

In [2]:
# Tensor 1: Mixed values
t1 = np.array([-1.5, -0.8, 0.0, 0.9, 2.3], dtype=np.float32)

# Tensor 2: Positive values
t2 = np.array([0.1, 0.5, 1.2, 2.0, 3.5], dtype=np.float32)

# Tensor 3: Negative values
t3 = np.array([-3.0, -2.1, -1.4, -0.6, -0.1], dtype=np.float32)

# Tensor 4: Constant values
t4 = np.array([5.0, 5.0, 5.0], dtype=np.float32)

# Tensor 5: Very small values
t5 = np.array([1e-9, 2e-9, -1e-9], dtype=np.float32)

tensors = {
    "Tensor 1": t1,
    "Tensor 2": t2,
    "Tensor 3": t3,
    "Tensor 4": t4,
    "Tensor 5": t5
}

## Step 2: Calculate Scale and Zero Point

The function:

- Finds the minimum and maximum tensor values.
- Computes the scale.
- Computes the zero point.
- Handles edge cases to avoid division by zero.

In [3]:
def calculate_scale_zero_point(tensor, q_min=-128, q_max=127):
    x_min = np.min(tensor)
    x_max = np.max(tensor)

    # Handle constant tensor
    if np.isclose(x_min, x_max):
        scale = 1.0
        zero_point = 0
        return scale, zero_point

    # Compute scale
    scale = (x_max - x_min) / (q_max - q_min)

    # Handle extremely small scale
    epsilon = 1e-12
    if scale < epsilon:
        scale = epsilon

    # Compute zero point
    zero_point = round(q_min - (x_min / scale))

    # Clip to INT8 range
    zero_point = int(np.clip(zero_point, q_min, q_max))

    return scale, zero_point

## Step 3: Quantization Function

Convert floating-point values into INT8 values.

In [4]:
def quantize_tensor(tensor, scale, zero_point, q_min=-128, q_max=127):

    quantized = np.round(tensor / scale) + zero_point

    quantized = np.clip(quantized, q_min, q_max)

    return quantized.astype(np.int8)

## Step 4: Dequantization Function

Convert INT8 values back to floating-point values.

In [5]:
def dequantize_tensor(q_tensor, scale, zero_point):

    return (q_tensor.astype(np.float32) - zero_point) * scale

## Step 5: Apply Quantization

For each tensor:

- Calculate scale
- Calculate zero point
- Quantize
- Dequantize
- Compute Mean Absolute Error (MAE)

In [6]:
for name, tensor in tensors.items():

    scale, zero_point = calculate_scale_zero_point(tensor)

    q_tensor = quantize_tensor(tensor, scale, zero_point)

    dq_tensor = dequantize_tensor(q_tensor, scale, zero_point)

    mae = np.mean(np.abs(tensor - dq_tensor))

    print("=" * 55)
    print(name)
    print("=" * 55)

    print("Tensor values:")
    print(tensor)

    print("\nTensor min:")
    print(np.min(tensor))

    print("\nTensor max:")
    print(np.max(tensor))

    print("\nScale:")
    print(scale)

    print("\nZero point:")
    print(zero_point)

    print("\nQuantized tensor:")
    print(q_tensor)

    print("\nDequantized tensor:")
    print(dq_tensor)

    print("\nMean Absolute Error:")
    print(mae)

    print("\n")

Tensor 1
Tensor values:
[-1.5 -0.8  0.   0.9  2.3]

Tensor min:
-1.5

Tensor max:
2.3

Scale:
0.01490196

Zero point:
-27

Quantized tensor:
[-128  -81  -27   33  127]

Dequantized tensor:
[-1.505098   -0.80470586  0.          0.8941176   2.2949018 ]

Mean Absolute Error:
0.004156864


Tensor 2
Tensor values:
[0.1 0.5 1.2 2.  3.5]

Tensor min:
0.1

Tensor max:
3.5

Scale:
0.013333334

Zero point:
-128

Quantized tensor:
[-120  -90  -38   22  127]

Dequantized tensor:
[0.10666667 0.50666666 1.2        2.         3.4       ]

Mean Absolute Error:
0.022666646


Tensor 3
Tensor values:
[-3.  -2.1 -1.4 -0.6 -0.1]

Tensor min:
-3.0

Tensor max:
-0.1

Scale:
0.011372549

Zero point:
127

Quantized tensor:
[-128  -58    4   74  118]

Dequantized tensor:
[-2.9        -2.1039217  -1.3988236  -0.6027451  -0.10235295]

Mean Absolute Error:
0.022039209


Tensor 4
Tensor values:
[5. 5. 5.]

Tensor min:
5.0

Tensor max:
5.0

Scale:
1.0

Zero point:
0

Quantized tensor:
[5 5 5]

Dequantized tensor:
[5

# Expected Observations

### Tensor 1 (Mixed values)

- Zero point lies near the center of the INT8 range.
- Quantization error is very small.

---

### Tensor 2 (Positive values)

- Zero point shifts toward the lower end of the INT8 range.
- Positive values utilize most of the available quantization levels.

---

### Tensor 3 (Negative values)

- Zero point shifts toward the upper end of the INT8 range.
- Negative values are represented efficiently.

---

### Tensor 4 (Constant Tensor)

- Since all values are identical, the range is zero.
- Scale is set to **1.0** to avoid division by zero.
- Quantization produces the same INT8 value for every element.

---

### Tensor 5 (Very Small Values)

- The computed scale is extremely small.
- Tiny floating-point differences are preserved as much as possible.
- Mean Absolute Error remains very low.